### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="hazelnut_spread_contaminant_detection",
    dataset_year="2020",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/45538",
    download_description="""
mkdir -p local-data-warehouse/hazelnut_spread_contaminant_detection/ && wget https://api.openml.org/data/download/22116506/dataset -O ./local-data-warehouse/hazelnut_spread_contaminant_detection/hazelnut.arff
""",
    # References
    academic_reference_bibtex=r"""@article{ricci2021machine,
  title={Machine-learning-based microwave sensing: A case study for the food industry},
  author={Ricci, Marco and {\v{S}}titi{\'c}, Bernardita and Urbinati, Luca and Di Guglielmo, Giuseppe and Vasquez, Jorge A Tob{\'o}n and Carloni, Luca P and Vipiana, Francesca and Casu, Mario R},
  journal={IEEE Journal on Emerging and Selected Topics in Circuits and Systems},
  volume={11},
  number={3},
  pages={503--514},
  year={2021},
  publisher={IEEE}
}
""",
    academic_reference_bibtex_key="ricci2021machine",
    license="CC BY-SA",
    data_tags=["IID"],
    curation_comments="""
- We select a features from a single frequency (10 GHz) as the authors also only considered this frequency for the final experiments.
- Anomaly: we use the publicly available dataset state, which is without preprocessing. Moreover, we were not able to inverse the original ordinal encoding of the label.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Contaminated",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Contaminated",
)

## Preprocessing

In [ ]:
import pandas as pd
import arff

with open(f"{dataset_mold.path}/hazelnut.arff") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = "Contaminated"
df = df.rename(columns={"class": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"}).astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()